# 01 — Data, Audit, Split Anti-Bocor, dan Data Sintetik

Notebook ini menjawab pertanyaan:

1. Data apa yang kita punya?
2. Apakah labelnya seimbang?
3. Apakah teks komposisinya panjang?
4. Apakah Knowledge Base setuju dengan gold label?
5. Bagaimana membagi data tanpa kebocoran?
6. Bagaimana data sintetik dibuat dan diaudit?

Output notebook ini dipakai oleh semua notebook training.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Notebook biasanya dijalankan dari folder notebooks/. Kita pindah ke root
# project agar semua path artefak konsisten.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "artifacts" / "data"
BERT_DIR = PROJECT_ROOT / "artifacts" / "bert" / "input"
RUNS_DIR = PROJECT_ROOT / "artifacts" / "notebook_runs"
for path in (DATA_DIR, BERT_DIR, RUNS_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Data dir     :", DATA_DIR)
print("BERT input   :", BERT_DIR)
print("Runs dir     :", RUNS_DIR)

In [ ]:
from app.experiment.configs import DataConfig
from app.experiment.data_pipeline import prepare_data, write_bert_inputs
from app.experiment.artifacts import write_json

pd.set_option("display.max_colwidth", 120)
print("Siap.")

## Langkah 1 — Baca CSV mentah

Kita baca apa adanya dulu. Delimiter dataset ini adalah titik koma (`;`),
bukan koma. Kalau salah delimiter, semua kolom akan terlihat menjadi satu.

In [ ]:
candidates = [
    PROJECT_ROOT / "ocr_output" / "data-mengandung.csv",
    PROJECT_ROOT.parent / "old" / "ocr_output" / "data-mengandung.csv",
    PROJECT_ROOT / "data.csv",
]
csv_path = next((path for path in candidates if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError("CSV tidak ditemukan. Set csv_path secara manual.")

df_raw = pd.read_csv(csv_path, delimiter=";", encoding="utf-8")
print("File :", csv_path)
print("Bentuk:", df_raw.shape)
print("Kolom :", list(df_raw.columns))
df_raw.head(5)

## Langkah 2 — Distribusi label

Dataset yang sehat untuk klasifikasi biner sebaiknya tidak terlalu timpang.
Kalau 95% data hanya satu kelas, model bisa "pintar" hanya dengan menebak kelas
mayoritas.

In [ ]:
label_counts = df_raw["label"].astype(str).str.lower().value_counts()
print(label_counts)

fig, ax = plt.subplots(figsize=(5, 3.5))
label_counts.plot(kind="bar", color=["#4C78A8", "#F58518"], ax=ax)
ax.set_xlabel("Label")
ax.set_ylabel("Jumlah sampel")
ax.set_title("Distribusi label dataset mentah")
for i, value in enumerate(label_counts.values):
    ax.text(i, value + 1, str(value), ha="center")
plt.tight_layout()
plt.show()

## Langkah 3 — Panjang teks komposisi

Panjang teks menentukan `max_len`. Kalau `max_len` terlalu kecil, banyak teks
terpotong. Kalau terlalu besar, training lebih lambat dan padding lebih banyak.

In [ ]:
from app.core.preprocessing.text import simple_tokenize

token_lengths = df_raw["text"].fillna("").astype(str).map(lambda t: len(simple_tokenize(t)))
stats = token_lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99])
print(stats)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(token_lengths, bins=40, color="#4C78A8", edgecolor="white")
ax.axvline(token_lengths.median(), color="black", linestyle="--", label=f"median={token_lengths.median():.0f}")
ax.axvline(token_lengths.quantile(0.95), color="red", linestyle=":", label=f"p95={token_lengths.quantile(0.95):.0f}")
ax.set_xlabel("Jumlah token per teks")
ax.set_ylabel("Frekuensi")
ax.legend()
plt.tight_layout()
plt.show()

## Langkah 4 — Gold label vs Knowledge Base

**Gold label** adalah acuan manusia. **KB** adalah aturan kata kunci yang
dibuat otomatis. KB berguna untuk:

- mengecek apakah aturan kita masuk akal,
- membuat data sintetik,
- menjelaskan error model.

Tetapi KB **tidak boleh menggantikan gold label**.

In [ ]:
config = DataConfig(
    csv_input=str(csv_path),
    output_dir=str(DATA_DIR),
    bert_input_dir=str(BERT_DIR),
    sort_source=True,   # split tidak berubah walau urutan baris CSV diacak
)
bundle = prepare_data(config)

# Simpan konfigurasi agar notebook lain memakai split yang sama.
write_json(DATA_DIR / "data_config.json", config.to_dict())

print("Ringkasan split:")
print(bundle.summary_frame().to_string(index=False))

## Langkah 5 — Audit split

Audit memastikan tidak ada produk atau teks yang sama muncul di dua split.
Kalau ada, model bisa "menghafal" jawaban dan skor menjadi tidak jujur.

In [ ]:
print("Audit leakage:")
print(bundle.split_audit.to_string(index=False))
print("\nKB vs Gold:")
print(bundle.kb_audit["summary"].to_string(index=False))
print("\nConfusion matrix KB vs Gold:")
print(bundle.kb_audit["confusion_matrix"].to_string())

## Langkah 6 — Data sintetik

Data sintetik dibuat **hanya dari real-train**, bukan dari validation/holdout.
Tujuannya menambah variasi komposisi. Setiap baris sintetik dicek ulang oleh
KB; kalau labelnya tidak konsisten, proses berhenti.

Perhatikan: data sintetik adalah **prior KB**, bukan data dunia nyata. Karena
itu kita selalu menyediakan opsi ablasi `synthetic_total=0`.

In [ ]:
print("Jumlah synthetic:", len(bundle.df_synthetic))
bundle.df_synthetic.head(8)[["text", "label"]]

## Langkah 7 — File split untuk IndoBERT

IndoBERT memakai file CSV yang sama persis dengan BiLSTM. Ini penting agar
perbandingan adil (apel vs apel). File yang dihasilkan:

- `train_real.csv` — 399 data real,
- `synthetic.csv` — data sintetik,
- `train_combined.csv` — real + sintetik (di-shuffle dengan seed),
- `val.csv` — 50 data validasi,
- `holdout.csv` — 50 data ujian akhir,
- `split_contract.json` — hash untuk memastikan file tidak berubah.

In [ ]:
contract = json.loads((BERT_DIR / "split_contract.json").read_text(encoding="utf-8"))
print(json.dumps(contract, indent=2, ensure_ascii=False))

## Checklist setelah notebook ini

- [ ] CSV berhasil dibaca dan punya kolom produk/teks/label.
- [ ] Distribusi label tidak ekstrem.
- [ ] Panjang teks masuk akal; `max_len` bisa dipilih.
- [ ] Audit leakage semua **PASS**.
- [ ] Tidak ada produk yang sama di train dan holdout.
- [ ] `split_contract.json` tersimpan.
- [ ] Notebook 02/03/04/05 memakai `data_config.json` yang sama.